# 01 - EDA

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Cari root repo: di laptop = folder di atas notebooks/, di Kaggle = dataset mlc-src
if os.path.exists("/kaggle/input"):
    hits = list(Path("/kaggle/input").rglob("src/data.py"))
    if hits:
        ROOT = hits[0].parent.parent
    else:  # dataset masih berupa zip
        ROOT = Path("/kaggle/working/_repo")
        for z in Path("/kaggle/input").rglob("*.zip"):
            if z.stem in ("src", "configs", "annotations"):
                zipfile.ZipFile(z).extractall(ROOT / z.stem)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import ON_KAGGLE, get_path, load_config
from src.seed import seed_everything

cfg = load_config()
seed_everything(cfg["SEED"])
print("Kaggle:", ON_KAGGLE, "| data:", get_path(cfg, "raw"))

Tujuan EDA ini:
1. Memahami struktur data train (video) dan test (sekuens frame).
2. Mengecek keseimbangan label, subjek, perilaku, dan sudut kamera.
3. Membandingkan train vs test (resolusi, jumlah frame, kecepatan gerak) supaya
   cara memotong video di `02_make_sequences` sesuai dengan bentuk data test.

In [ ]:
import re
import glob

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 50)

RAW = get_path(cfg, "raw")
TRAIN_DIR, TEST_DIR = RAW / "train", RAW / "test"
for p in sorted(RAW.iterdir()):
    print(p.name)

## 1. Data train: indeks video

In [ ]:
def video_meta(path):
    cap = cv2.VideoCapture(str(path))
    meta = {
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "n_frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    return meta


rows = []
for f in tqdm(sorted(TRAIN_DIR.glob("*/*/*")), desc="video"):
    if not f.is_file():
        continue
    view, behavior = f.stem.split("_", 1)  # front_Looking_Under_the_Desk
    rows.append({
        "label_name": f.parent.parent.name,          # cheat / normal
        "subject": f.parent.name,                    # subject_1
        "subject_id": int(re.findall(r"\d+", f.parent.name)[0]),
        "view": view,                                # front / side
        "behavior": behavior.replace("_", " "),
        "path": str(f.relative_to(RAW)),
        **video_meta(f),
    })

train_df = pd.DataFrame(rows)
train_df["label"] = (train_df["label_name"] == "cheat").astype(int)
train_df["duration_s"] = train_df["n_frames"] / train_df["fps"]
print(train_df.shape)
train_df.head()

In [ ]:
print("Jumlah video per label:")
display(train_df["label_name"].value_counts())

print("Perilaku per label:")
display(pd.crosstab(train_df["behavior"], train_df["label_name"]))

print("Sudut kamera per label:")
display(pd.crosstab(train_df["view"], train_df["label_name"]))

In [ ]:
# Apakah setiap subjek lengkap? (harusnya 8 cheat + 8 normal)
per_subject = pd.crosstab(train_df["subject_id"], train_df["label_name"]).sort_index()
print("Subjek:", per_subject.shape[0])
print("Subjek yang tidak lengkap:")
display(per_subject[(per_subject != per_subject.mode().iloc[0]).any(axis=1)])

In [ ]:
display(train_df[["fps", "n_frames", "width", "height", "duration_s"]].describe().round(2))
print("Resolusi unik:", train_df.groupby(["width", "height"]).size().to_dict())
print("FPS unik:", train_df["fps"].round(2).value_counts().to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for name, g in train_df.groupby("label_name"):
    axes[0].hist(g["duration_s"], bins=20, alpha=0.6, label=name)
axes[0].set(title="Durasi video train", xlabel="detik", ylabel="jumlah video")
axes[0].legend()

train_df.groupby("behavior")["duration_s"].mean().sort_values().plot.barh(ax=axes[1])
axes[1].set(title="Rata-rata durasi per perilaku", xlabel="detik", ylabel="")
plt.tight_layout()
plt.show()

## 2. Data test: sekuens frame

In [ ]:
rows = []
for d in tqdm(sorted(p for p in TEST_DIR.iterdir() if p.is_dir()), desc="seq"):
    frames = sorted(d.glob("*.jpg"))
    h, w = cv2.imread(str(frames[0])).shape[:2]  # cukup frame pertama
    rows.append({"id": d.name, "n_frames": len(frames), "width": w, "height": h})

test_df = pd.DataFrame(rows)
print(test_df.shape)
display(test_df.describe().round(2))
print("Jumlah frame unik:", test_df["n_frames"].value_counts().to_dict())
print("Resolusi unik:", test_df.groupby(["width", "height"]).size().to_dict())

In [ ]:
sub = pd.read_csv(RAW / "sample_submission.csv")
print(sub.shape)
display(sub.head())
print("Kolom:", list(sub.columns))
print("ID submission == folder test:", set(sub["id"]) == set(test_df["id"]))

## 3. Contoh visual

In [ ]:
def read_frame(path, idx):
    cap = cv2.VideoCapture(str(RAW / path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else None


# Satu frame (tengah video) untuk setiap perilaku, subjek pertama, kamera front & side
subj = train_df["subject"].iloc[0]
sample = (train_df[train_df["subject"] == subj]
          .sort_values(["label_name", "behavior", "view"]))
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(read_frame(r["path"], r["n_frames"] // 2))
    ax.set_title(f"{r['label_name']} | {r['behavior']}\n{r['view']}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Train - {subj}")
plt.tight_layout()
plt.show()

In [ ]:
# Beberapa sekuens test: frame ke-1, 10, 20, 30
show_ids = test_df["id"].sample(4, random_state=cfg["SEED"]).tolist()
fig, axes = plt.subplots(len(show_ids), 4, figsize=(14, 2.3 * len(show_ids)))
for row, sid in zip(axes, show_ids):
    frames = sorted((TEST_DIR / sid).glob("*.jpg"))
    for ax, i in zip(row, [0, 9, 19, 29]):
        ax.imshow(cv2.cvtColor(cv2.imread(str(frames[i])), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{sid} | frame {i + 1}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train vs test: kecepatan gerak antar-frame

Test hanya 30 frame per sekuens. Kalau frame test diambil berurutan dari video 30 fps,
satu sekuens = 1 detik. Kalau diambil dengan loncatan (stride), satu sekuens lebih panjang.

Cara mengecek: hitung rata-rata perbedaan piksel antar-frame berurutan di test, lalu
bandingkan dengan perbedaan frame train yang berjarak `k` frame (k = 1, 2, 3, ...).
Nilai `k` yang paling mirip dengan test kira-kira adalah stride yang dipakai penyelenggara.

In [ ]:
SIZE = (test_df["width"].iloc[0], test_df["height"].iloc[0])  # samakan ke resolusi test


def gray(img):
    return cv2.cvtColor(cv2.resize(img, SIZE), cv2.COLOR_BGR2GRAY)  # uint8, hemat RAM


def diff(a, b):
    return cv2.absdiff(a, b).mean()


def test_motion(sid):
    fr = [gray(cv2.imread(str(p))) for p in sorted((TEST_DIR / sid).glob("*.jpg"))]
    return np.mean([diff(a, b) for a, b in zip(fr[:-1], fr[1:])])


def train_motion(path, ks, n_pairs=20):
    cap = cv2.VideoCapture(str(RAW / path))
    fr = []
    ok, f = cap.read()
    while ok:
        fr.append(gray(f))
        ok, f = cap.read()
    cap.release()
    starts = np.linspace(0, len(fr) - max(ks) - 1, n_pairs).astype(int)
    return {k: np.mean([diff(fr[s], fr[s + k]) for s in starts]) for k in ks}


KS = [1, 2, 3, 4, 5, 6, 8, 10, 15]
rng = np.random.default_rng(cfg["SEED"])
test_mot = [test_motion(s) for s in tqdm(rng.choice(test_df["id"], 60, replace=False), desc="test")]
vids = train_df.groupby(["label_name", "view"]).sample(3, random_state=cfg["SEED"])["path"]
train_mot = pd.DataFrame([train_motion(p, KS) for p in tqdm(vids, desc="train")])

print(f"Test  (frame berurutan): median {np.median(test_mot):.2f}")
print("Train (jarak k frame)  :")
print(train_mot.median().round(2).to_string())

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(KS, train_mot.median(), "o-", label="train, jarak k frame")
plt.axhline(np.median(test_mot), color="C1", ls="--", label="test, frame berurutan")
plt.xlabel("k (jarak frame di video train)")
plt.ylabel("rata-rata |beda piksel|")
plt.legend()
plt.title("Perkiraan stride frame test")
plt.show()

## 5. Ringkasan temuan

Hasil dari menjalankan notebook ini:

- **Train:** 416 video `.MOV` dari 26 subjek. Setiap subjek lengkap: 16 video
  (4 perilaku cheat + 4 perilaku normal) × 2 kamera (front, side).
- **Label seimbang:** 208 cheat, 208 normal. Front/side juga seimbang (104 per label).
- **Perilaku menentukan label:**
  - cheat = Looking Under the Desk, Reading Hidden Notes, Repeated Peeking, Using Smartwatch
  - normal = Checking Smartwatch, Looking Around, Thinking, Writing Answers
  - Pasangan yang mirip dan kemungkinan sulit dibedakan: *Using* vs *Checking Smartwatch*,
    *Repeated Peeking* vs *Looking Around*.
- **Video train:** 1280×720, ±30 fps, 407–558 frame (13,6–18,6 detik, rata-rata 16 detik).
- **Test:** 1515 sekuens, semua tepat **30 frame**, resolusi **640×360** (setengah train).
  ID di `sample_submission.csv` (kolom `id,label`) cocok dengan folder test.
- **Perkiraan stride frame test:** beda piksel antar-frame test (0,94) ada di antara
  train k=1 (0,75) dan k=2 (1,05), jadi frame test kemungkinan **berurutan atau loncat 1–2 frame**
  (1 sekuens ≈ 1–2 detik). Kompresi JPEG di test sedikit menaikkan angka ini, jadi k=1 juga masuk akal.
  → Di `02_make_sequences`: potong video train jadi jendela `SEQ_LEN=30` frame dengan stride 1
  (coba juga 2), lalu resize ke 640×360.
- **Validasi:** split harus per **subjek** (`annotations/folds.csv`), supaya subjek yang sama
  tidak muncul di train dan validasi sekaligus. Jendela dari video yang sama juga sangat mirip,
  jadi jangan diacak per jendela.
